In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEndpoint ,ChatHuggingFace
from langchain_core.utils.function_calling import convert_to_openai_tool
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
from dotenv import load_dotenv
load_dotenv() 

In [ ]:
llm = HuggingFaceEndpoint( model="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation"
                          )

model = ChatHuggingFace(llm=llm)

In [ ]:
class sentiment_finding(BaseModel):
    sentiment: Literal["positive","negative",] = Field(description="The sentiment of the review")

schema_dict = convert_to_openai_tool(sentiment_finding)
structured_model = model.with_structured_output(schema_dict)
  

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal[
        "UX",
        "Performance",
        "Bug",
        "Support",
        "Other"
    ] = Field(
        description="The category of the user's issue."
    )

    tone: Literal[
        "angry",
        "frustrated",
        "disappointed",
        "calm"
    ] = Field(
        description="The emotional tone expressed by the user."
    )

    urgency: Literal[
        "low",
        "medium",
        "high"
    ] = Field(
        description="How urgent or critical the issue is."
    )

schema_dict2 = convert_to_openai_tool(DiagnosisSchema)
structured_model2 = model.with_structured_output(schema_dict2)    

In [ ]:
class reviewstate(TypedDict):
    review:str
    sentiment:Literal["positive","negative",] 
    Diagnoses: dict
    response:str
    

In [ ]:

# prompt = f"Analyze the following review and determine its sentiment (positive, negative):\n\n{review}"
# result = structured_model.invoke(prompt)
# validated = sentiment_finding.model_validate(result)
# print(type(validated))

In [ ]:
def sentiment_analysis(state: reviewstate):
    prompt = f"Analyze the following review and determine its sentiment (positive, negative):\n\n{state['review']}"

    result = structured_model.invoke(prompt)
    validated = sentiment_finding.model_validate(result)

    return {"sentiment": validated.sentiment}

def cheek_sentiment(state: reviewstate)-> Literal["positive","run_dig"]:
    if state['sentiment'] == "positive":
        return "positive"
    else:
        return "run_dig"

    
def positive_response(state: reviewstate):
    prompt = f"Generate a positive response to the following review:\n\n{state['review']}"
    response = model.invoke(prompt)
    return {"response": response.content}

def run_dig(state: reviewstate):
    prompt = f"""Diagnose this negative review: \n\n{state['review']}.\n\n Return tone, Urgency, and issue type in a structured format."""
    result2 = structured_model2.invoke(prompt)
    validated2 = DiagnosisSchema.model_validate(result2)
    return {"Diagnoses": validated2.model_dump()}

def negative_response(state: reviewstate):
    diagnosis = state['Diagnoses']
    prompt = f"""You are a support assistant.The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.Write an empathetic, helpful resolution messag"""
    response = model.invoke(prompt)
    return {"response": response.content}
    
     







In [ ]:
graph = StateGraph(reviewstate)
#define nodes
graph.add_node("sentiment_analysis", sentiment_analysis)
graph.add_node("positive_response", positive_response)
graph.add_node("run_dig", run_dig)
graph.add_node("negative_response", negative_response)

# Define edges
graph.add_edge(START, "sentiment_analysis")
graph.add_conditional_edges(
    "sentiment_analysis",
    cheek_sentiment,
    {
        "positive": "positive_response",
        "run_dig": "run_dig",
    },
)
graph.add_edge( "positive_response",END)
graph.add_edge("run_dig", "negative_response")
graph.add_edge("negative_response", END)
workflow=graph.compile()

In [ ]:
input_review = {"review": "This product is a complete waste of money. It broke after two days of light use, customer support never responded to my emails, and the whole experience has been incredibly frustrating. I want a refund immediately."
}
result = workflow.invoke(input_review)


In [ ]:
print(result)